# vi-function-calling-slm · 04 xuất GGUF + đánh giá

Kaggle **GPU T4** (Unsloth không chạy trên P100), bật Internet.
Input: code + dữ liệu (notebook 01) + output notebook 03 (`sft-merged`, `dpo-merged`).
Secrets: `HF_TOKEN` (đẩy GGUF), `GEMINI_API_KEY` (cận trên, chỉ để đánh giá).

1. Xuất SFT+DPO sang GGUF `q4_k_m` (~1.1GB) bằng `save_pretrained_gguf` của Unsloth
2. Đánh giá trên tập test tổng hợp + bộ viết tay:
   - Qwen3-1.7B gốc (chỉ prompt, cùng system prompt + tools)
   - SFT, SFT+DPO (transformers fp16 trên GPU)
   - SFT+DPO GGUF Q4_K_M qua llama.cpp (CPU) — bản thật sự chạy trên laptop, để xem lượng tử hoá mất bao nhiêu
   - Gemini (function calling qua API) làm cận trên
3. Bảng `results/eval.md`

Độ trễ ở đây đo trên CPU Kaggle, không đại diện cho laptop (đo lại bằng `vi_fc.evaluate predict` trên máy thật).

In [ ]:
import gc, glob, json, os, shutil, sys

CODE = glob.glob("/kaggle/input/**/vi_fc/__init__.py", recursive=True)[0].rsplit("/vi_fc", 1)[0]
REPO = CODE.rsplit("/src", 1)[0]
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
DATA = os.path.dirname(glob.glob("/kaggle/input/**/gen/test.jsonl", recursive=True)[0])
SFT = os.path.dirname(glob.glob("/kaggle/input/**/sft-merged/config.json", recursive=True)[0])
DPO = os.path.dirname(glob.glob("/kaggle/input/**/dpo-merged/config.json", recursive=True)[0])
print(CODE, DATA, SFT, DPO)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q unsloth python-dotenv
# wheel CPU dựng sẵn, khỏi build llama.cpp từ source (~15 phút)
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

from kaggle_secrets import UserSecretsClient

sec = UserSecretsClient()
for k in ("HF_TOKEN", "GEMINI_API_KEY"):
    try:
        os.environ[k] = sec.get_secret(k)
    except Exception:
        print("thiếu secret", k)

## Xuất GGUF q4_k_m

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(DPO, max_seq_length=3072, load_in_4bit=False,
                                                     dtype=torch.float16)
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
del model
gc.collect(); torch.cuda.empty_cache()
GGUF = sorted(glob.glob("gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf"))
print(GGUF)
GGUF = [g for g in GGUF if "q4_k_m" in g.lower()][0]
!ls -la {GGUF}

In [ ]:
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi

    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo("DuongCodeAI/vi-fc-qwen3-1.7b-GGUF", exist_ok=True)
    api.upload_file(path_or_fileobj=GGUF, path_in_repo="vi-fc-qwen3-1.7b-Q4_K_M.gguf",
                    repo_id="DuongCodeAI/vi-fc-qwen3-1.7b-GGUF")

## Đánh giá

In [ ]:
from vi_fc.evaluate import load_jsonl, predict, report
from vi_fc.inference import FunctionCaller

SETS = {"synth_test": load_jsonl(f"{DATA}/test.jsonl"), "manual": load_jsonl(f"{REPO}/data/test_manual.jsonl")}
os.makedirs("results/preds", exist_ok=True)
print({k: len(v) for k, v in SETS.items()})


def run(system, fc, sleep_s=0.0):
    for name, recs in SETS.items():
        predict(recs, fc, system, name, f"results/preds/{system}_{name}.jsonl", sleep_s)
        print("xong", system, name)


for system, path in (("qwen3-1.7b-base", "Qwen/Qwen3-1.7B"), ("sft", SFT), ("sft-dpo", DPO)):
    fc = FunctionCaller("hf", path, guard=False)
    run(system, fc)
    del fc
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
fc = FunctionCaller("llama_cpp", GGUF, n_threads=os.cpu_count(), guard=False)
print("warmup", fc.warmup())
run("sft-dpo-q4km", fc)

In [ ]:
if os.environ.get("GEMINI_API_KEY"):
    run("gemini", FunctionCaller("gemini", guard=False), sleep_s=4.0)  # free tier ~15 request/phút

In [ ]:
res = report(sorted(glob.glob("results/preds/*.jsonl")), "results")
print(open("results/eval.md", encoding="utf-8").read())

## Câu sai của bản GGUF trên bộ viết tay

In [ ]:
from vi_fc.match import actions_match
from vi_fc.parse import Action

for r in load_jsonl("results/preds/sft-dpo-q4km_manual.jsonl"):
    if not actions_match(Action.from_dict(r["gold"]), Action.from_dict(r["pred"])):
        print(r["id"], r["user"]); print("   gold:", r["gold"]); print("   raw :", r["raw"][:200])